In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

# Read source file
customers_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset/olist_customers_dataset.csv")
)

# # Create a new customer that does not exist in Bronze
dummy_customer = spark.createDataFrame([
    Row(
        customer_id="INC_TEST_001",
        customer_unique_id="INC_TEST_001",
        customer_zip_code_prefix=111111,
        customer_city="PUNE",
        customer_state="MH"
    )
])

# # # Match datatype with source
dummy_customer = dummy_customer.withColumn(
    "customer_zip_code_prefix",
    F.col("customer_zip_code_prefix").cast("int")
)

# # Add dummy record to source data
customers_incremental_df = customers_df.unionByName(dummy_customer)

# # Existing Bronze table
existing_df = spark.table("bronze.customers")

# # Incremental logic:
# # Keep only records whose customer_id is not already in Bronze
new_records = customers_incremental_df.join(
    existing_df.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("New records found:", new_records.count())

# # View the records that will be inserted
new_records.show(truncate=False)

# # Append only new records
(
    new_records.write
               .format("delta")
               .mode("append")
               .saveAsTable("bronze.customers")
)
print("Incremental load completed")